# CSV → Parquet Conversion

Converts raw CSV datasets to parquet. Shared imports are at the top; each dataset has its own section.

Output layout:
```
data/parquet/
├── daily/fleet-daily-YYYY-MM.parquet      (60 files, one per month)
├── monthly/fleet-monthly-YYYY.parquet     (5 files, one per year)
└── mmsi-daily/mmsi-daily-YYYY.parquet     (5 files, one per year)
```

In [ ]:

import glob
import os
from pathlib import Path
from itertools import groupby

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

import geopandas as gpd

## Fleet Daily

One parquet per calendar month → `data/parquet/daily/fleet-daily-YYYY-MM.parquet`

In [ ]:
DATA_DIR = Path("../data")
OUTPUT_DIR = DATA_DIR / "parquet" / "daily"
OUTPUT_DIR.mkdir(exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024]

# Parquet schema with downcasted types
ARROW_SCHEMA = pa.schema([
    pa.field("date",          pa.date32()),
    pa.field("cell_ll_lat",   pa.float32()),
    pa.field("cell_ll_lon",   pa.float32()),
    pa.field("flag",          pa.dictionary(pa.int16(), pa.string())),
    pa.field("geartype",      pa.dictionary(pa.int8(),  pa.string())),
    pa.field("hours",         pa.float32()),
    pa.field("fishing_hours", pa.float32()),
    pa.field("mmsi_present",  pa.int16()),
])

# pandas dtypes used when reading each CSV
CSV_DTYPES = {
    "cell_ll_lat":   "float32",
    "cell_ll_lon":   "float32",
    "flag":          "category",
    "geartype":      "category",
    "hours":         "float32",
    "fishing_hours": "float32",
    "mmsi_present":  "int16",
}

In [ ]:
def normalize_flag(flag: pd.Series) -> pd.Series:
    """Collapse GFW UNKNOWN-<ISO> flags into <ISO> (bare UNKNOWN is kept).

    GFW assigns flag UNKNOWN-<ISO> to invalid-MID vessels that spend >50% of
    their fishing hours in the <ISO> EEZ (see README-known-issues-v3.txt). Per
    GFW's own docs this is most likely <ISO>-flagged activity broadcasting a
    wrong MID, so we merge it into <ISO>.
    """
    return (
        flag.astype("string")
        .str.replace(r"^UNKNOWN-(?=.)", "", regex=True)
        .astype("category")
    )


In [ ]:
def csv_path_to_year_month(path: Path) -> tuple[int, int]:
    """Extract (year, month) from a fleet CSV filename."""
    # filename: fleet-daily-csvs-100-v3-YYYY-MM-DD.csv
    date_part = path.stem.split("-v3-")[1]   # YYYY-MM-DD
    year, month, _ = date_part.split("-")
    return int(year), int(month)


def convert_month(csv_paths: list[Path], output_path: Path) -> int:
    """
    Stream-convert a list of daily CSVs (one month) into a single parquet file.
    Returns total row count written.
    """
    frames = []
    for csv_path in sorted(csv_paths):
        df = pd.read_csv(
            csv_path,
            dtype=CSV_DTYPES,
            parse_dates=["date"],
        )
        frames.append(df)

    month_df = pd.concat(frames, ignore_index=True)
    month_df["flag"] = normalize_flag(month_df["flag"])

    table = pa.Table.from_pandas(month_df, schema=ARROW_SCHEMA, preserve_index=False)
    pq.write_table(table, output_path, compression="snappy")

    return len(month_df)


def convert_year(year: int, skip_existing: bool = True) -> None:
    """Convert all daily CSVs for a given year into monthly parquet files."""
    folder = DATA_DIR / f"fleet-daily-{year}"
    csv_paths = sorted(folder.glob("fleet-daily-csvs-100-v3-*.csv"))

    if not csv_paths:
        print(f"  [!] No CSVs found in {folder}")
        return

    # Group by month
    def month_key(p):
        return csv_path_to_year_month(p)[1]

    for month, group in groupby(csv_paths, key=month_key):
        output_path = OUTPUT_DIR / f"fleet-daily-{year}-{month:02d}.parquet"

        if skip_existing and output_path.exists():
            print(f"  skip  {output_path.name} (already exists)")
            continue

        paths = list(group)
        n_rows = convert_month(paths, output_path)
        size_mb = output_path.stat().st_size / 1e6
        print(f"  wrote {output_path.name}  ({len(paths)} days, {n_rows:,} rows, {size_mb:.1f} MB)")

In [ ]:
for year in YEARS:
    print(f"\n=== {year} ===")
    convert_year(year)

## Verification

Sanity-check a sample parquet file: schema, row count, and a preview.

In [ ]:
sample = OUTPUT_DIR / "fleet-daily-2020-01.parquet"

meta = pq.read_metadata(sample)
print(f"File:       {sample.name}")
print(f"Rows:       {meta.num_rows:,}")
print(f"Row groups: {meta.num_row_groups}")
print(f"Size:       {sample.stat().st_size / 1e6:.1f} MB")
print()
print(pq.read_schema(sample))

df_sample = pd.read_parquet(sample)
print()
print(df_sample.dtypes)
print()
df_sample.head()

In [ ]:
# Summary: output file sizes across all years
parquet_files = sorted(OUTPUT_DIR.glob("fleet-daily-*.parquet"))
total_mb = sum(f.stat().st_size for f in parquet_files) / 1e6
print(f"{'File':<35} {'MB':>8}")
print("-" * 44)
for f in parquet_files:
    print(f"  {f.name:<33} {f.stat().st_size / 1e6:>7.1f}")
print("-" * 44)
print(f"  {'TOTAL':<33} {total_mb:>7.1f}")

## Fleet Monthly

One parquet per year → `data/parquet/monthly/fleet-monthly-YYYY.parquet`

In [ ]:
MONTHLY_OUTPUT_DIR = DATA_DIR / "parquet" / "monthly"
MONTHLY_OUTPUT_DIR.mkdir(exist_ok=True)

MONTHLY_ARROW_SCHEMA = pa.schema([
    pa.field("date",          pa.date32()),
    pa.field("year",          pa.int16()),
    pa.field("month",         pa.int8()),
    pa.field("cell_ll_lat",   pa.float32()),
    pa.field("cell_ll_lon",   pa.float32()),
    pa.field("flag",          pa.dictionary(pa.int16(), pa.string())),
    pa.field("geartype",      pa.dictionary(pa.int8(),  pa.string())),
    pa.field("hours",         pa.float32()),
    pa.field("fishing_hours", pa.float32()),
    pa.field("mmsi_present",  pa.int16()),
])

MONTHLY_CSV_DTYPES = {
    "year":          "int16",
    "month":         "int8",
    "cell_ll_lat":   "float32",
    "cell_ll_lon":   "float32",
    "flag":          "category",
    "geartype":      "category",
    "hours":         "float32",
    "fishing_hours": "float32",
    "mmsi_present":  "int16",
}

In [ ]:
def convert_fleet_monthly_year(year: int, skip_existing: bool = True) -> None:
    """Convert all monthly CSVs for a given year into a single parquet file."""
    folder = DATA_DIR / f"fleet-monthly-{year}"
    csv_paths = sorted(folder.glob("fleet-monthly-csvs-10-v3-*.csv"))

    if not csv_paths:
        print(f"  [!] No CSVs found in {folder}")
        return

    output_path = MONTHLY_OUTPUT_DIR / f"fleet-monthly-{year}.parquet"

    if skip_existing and output_path.exists():
        print(f"  skip  {output_path.name} (already exists)")
        return

    frames = []
    for csv_path in csv_paths:
        df = pd.read_csv(csv_path, dtype=MONTHLY_CSV_DTYPES, parse_dates=["date"])
        frames.append(df)

    year_df = pd.concat(frames, ignore_index=True)
    year_df["flag"] = normalize_flag(year_df["flag"])
    table = pa.Table.from_pandas(year_df, schema=MONTHLY_ARROW_SCHEMA, preserve_index=False)
    pq.write_table(table, output_path, compression="snappy")

    size_mb = output_path.stat().st_size / 1e6
    print(f"  wrote {output_path.name}  ({len(csv_paths)} months, {len(year_df):,} rows, {size_mb:.1f} MB)")

In [ ]:
for year in YEARS:
    convert_fleet_monthly_year(year)

In [ ]:
sample = MONTHLY_OUTPUT_DIR / "fleet-monthly-2020.parquet"
meta = pq.read_metadata(sample)
print(f"Rows: {meta.num_rows:,}  |  Size: {sample.stat().st_size / 1e6:.1f} MB")
pd.read_parquet(sample).head()

## MMSI Daily

One parquet per year → `data/parquet/mmsi-daily/mmsi-daily-YYYY.parquet`

Uses `ParquetWriter` to write month-by-month, keeping peak RAM to ~one month of data.

In [ ]:
MMSI_OUTPUT_DIR = DATA_DIR / "parquet" / "mmsi-daily"
MMSI_OUTPUT_DIR.mkdir(exist_ok=True)

MMSI_ARROW_SCHEMA = pa.schema([
    pa.field("date",          pa.date32()),
    pa.field("cell_ll_lat",   pa.float32()),
    pa.field("cell_ll_lon",   pa.float32()),
    pa.field("mmsi",          pa.string()),
    pa.field("hours",         pa.float32()),
    pa.field("fishing_hours", pa.float32()),
])

MMSI_CSV_DTYPES = {
    "cell_ll_lat":   "float32",
    "cell_ll_lon":   "float32",
    "mmsi":          "str",
    "hours":         "float32",
    "fishing_hours": "float32",
}

In [ ]:
def convert_mmsi_daily_year(year: int, skip_existing: bool = True) -> None:
    """
    Convert all daily MMSI CSVs for a given year into a single parquet file.
    Writes month-by-month via ParquetWriter to cap peak RAM at ~one month of data.
    """
    folder = DATA_DIR / f"mmsi-daily-{year}"
    csv_paths = sorted(folder.glob("mmsi-daily-csvs-10-v3-*.csv"))

    if not csv_paths:
        print(f"  [!] No CSVs found in {folder}")
        return

    output_path = MMSI_OUTPUT_DIR / f"mmsi-daily-{year}.parquet"

    if skip_existing and output_path.exists():
        print(f"  skip  {output_path.name} (already exists)")
        return

    # csv_path_to_year_month works here too: splits on "-v3-" → "YYYY-MM-DD"
    def month_key(p):
        return csv_path_to_year_month(p)[1]

    total_rows = 0
    with pq.ParquetWriter(output_path, MMSI_ARROW_SCHEMA, compression="snappy") as writer:
        for month, group in groupby(csv_paths, key=month_key):
            frames = [
                pd.read_csv(p, dtype=MMSI_CSV_DTYPES, parse_dates=["date"])
                for p in group
            ]
            month_df = pd.concat(frames, ignore_index=True)
            table = pa.Table.from_pandas(month_df, schema=MMSI_ARROW_SCHEMA, preserve_index=False)
            writer.write_table(table)
            total_rows += len(month_df)
            print(f"    {year}-{month:02d}  {len(month_df):,} rows")

    size_mb = output_path.stat().st_size / 1e6
    print(f"  wrote {output_path.name}  ({len(csv_paths)} days, {total_rows:,} rows, {size_mb:.1f} MB)")

In [ ]:
for year in YEARS:
    print(f"\n=== {year} ===")
    convert_mmsi_daily_year(year)

In [ ]:
sample = MMSI_OUTPUT_DIR / "mmsi-daily-2020.parquet"
meta = pq.read_metadata(sample)
print(f"Rows: {meta.num_rows:,}  |  Size: {sample.stat().st_size / 1e6:.1f} MB")
pd.read_parquet(sample).head()

# EEZ

In [ ]:
eez = gpd.read_file(DATA_DIR / "World_EEZ_v12_20231025" / "eez_v12.shp")

### 📍 Understanding EEZ Shapefile Data

This dataset is not a standard CSV but a **geospatial dataset (Shapefile)** loaded as a `GeoDataFrame`.  
You can think of it as a regular table with an additional **geometry column**.

#### Structure of the data
- Each **row** represents one Exclusive Economic Zone (EEZ), i.e. a maritime area associated with a country or territory.
- The dataset contains both **attributes** (like a CSV) and **geometric shapes**.

#### Key columns
- `geometry`: the most important column — it contains the **polygon** (or multipolygon) defining the spatial boundaries of the EEZ.
- `ISO_TER1`: ISO country/territory code (e.g. `FRA`, `USA`)
- `TERRITORY1`: name of the territory
- `SOVEREIGN1`: sovereign country

#### Important note
Some rows correspond to **overlapping or disputed zones** (e.g. "Overlapping claim"), which can be filtered depending on the analysis.

#### Conceptual view
- AIS dataset → points (latitude, longitude)
- EEZ dataset → polygons (areas)

This allows performing **spatial joins** to determine in which EEZ each vessel is located.

In [ ]:
print(eez.head())

In [ ]:
print(eez.crs)

In [ ]:
from shapely.geometry import box

# =========================================
# Geographic focus: Taiwan / South-East China Sea
# =========================================

MIN_LON, MAX_LON = 110.0, 127.5
MIN_LAT, MAX_LAT = 8.0, 29.0

TAIWAN_BBOX = box(MIN_LON, MIN_LAT, MAX_LON, MAX_LAT)


def load_clean_eez_taiwan_region() -> gpd.GeoDataFrame:
    """
    Load EEZ polygons, keep standard 200NM EEZ only,
    and restrict them to the Taiwan / East Asia region.
    """
    eez = gpd.read_file(EEZ_PATH)
    eez = eez[eez["POL_TYPE"] == "200NM"].copy()

    eez = eez[
        ["GEONAME", "ISO_TER1", "TERRITORY1", "SOVEREIGN1", "geometry"]
    ].rename(
        columns={
            "GEONAME": "eez_name",
            "ISO_TER1": "eez_iso",
            "TERRITORY1": "eez_territory",
            "SOVEREIGN1": "eez_sovereign",
        }
    )

    region_gdf = gpd.GeoDataFrame(
        {"geometry": [TAIWAN_BBOX]},
        crs="EPSG:4326"
    )

    eez_region = gpd.overlay(eez, region_gdf, how="intersection")
    return eez_region


EEZ_GDF = load_clean_eez_taiwan_region()


def add_eez_to_dataframe_region(df: pd.DataFrame, eez_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Restrict points to the Taiwan region first, then assign EEZ information.
    Points outside any EEZ in the selected region are labeled as High Seas.
    Points outside the region are dropped.
    """
    df_region = df[
        (df["cell_ll_lon"] >= MIN_LON) & (df["cell_ll_lon"] <= MAX_LON) &
        (df["cell_ll_lat"] >= MIN_LAT) & (df["cell_ll_lat"] <= MAX_LAT)
    ].copy()

    points_gdf = gpd.GeoDataFrame(
        df_region,
        geometry=gpd.points_from_xy(df_region["cell_ll_lon"], df_region["cell_ll_lat"]),
        crs="EPSG:4326"
    )

    joined = gpd.sjoin(
        points_gdf,
        eez_gdf,
        how="left",
        predicate="within"
    )

    joined["maritime_zone"] = joined["eez_name"].fillna("High Seas")
    joined["eez_iso"] = joined["eez_iso"].fillna("INT")
    joined["eez_territory"] = joined["eez_territory"].fillna("High Seas")
    joined["eez_sovereign"] = joined["eez_sovereign"].fillna("International Waters")

    joined = joined.drop(columns=["geometry", "index_right"], errors="ignore")

    return pd.DataFrame(joined)

In [ ]:
sample_path = OUTPUT_DIR / "fleet-daily-2020-01.parquet"
df = pd.read_parquet(sample_path)

df_taiwan = add_eez_to_dataframe_region(df, EEZ_GDF)

print(df_taiwan.head())
df_taiwan.head()

In [ ]:
# =========================================
# Parameters
# =========================================
country_col = "flag"      # change if needed
vessel_id_col = "mmsi_present"
top_n = 8
year_label = "2024"       # change if needed

# =========================================
# Compute number of unique boats by country
# =========================================
boats_by_country = (
    df_taiwan.groupby(country_col)[vessel_id_col]
    .nunique()
    .sort_values(ascending=False)
)

total_boats = boats_by_country.sum()

# Keep top N, group the rest as "Other"
top = boats_by_country.iloc[:top_n].copy()
other = boats_by_country.iloc[top_n:].sum()

if other > 0:
    top["Other"] = other

percentages = 100 * top / total_boats

# Labels for legend
legend_labels = [
    f"{country} ({pct:.1f}%)"
    for country, pct in zip(top.index, percentages)
]

# Show percentages only for large enough slices
def autopct_func(pct):
    return f"{pct:.0f}%" if pct >= 4 else ""

# =========================================
# Plot
# =========================================
fig, ax = plt.subplots(figsize=(10, 6))

wedges, _, _ = ax.pie(
    top.values,
    startangle=90,
    counterclock=False,
    wedgeprops=dict(width=0.40, edgecolor="white", linewidth=2),
    autopct=autopct_func,
    pctdistance=0.80,
    textprops=dict(color="#2b2b2b", fontsize=12, weight="bold")
)

# Center text
ax.text(0, 0.10, year_label, ha="center", va="center",
        fontsize=22, fontweight="bold", color="#2b2b2b")
ax.text(0, -0.08, f"{total_boats:,} boats", ha="center", va="center",
        fontsize=14, color="#5a5a5a")

# Title
ax.set_title("Boats by Country", fontsize=24, fontweight="bold", pad=20)

# Legend
ax.legend(
    wedges,
    legend_labels,
    title="Country",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=12,
    title_fontsize=14
)

# Footnote
fig.text(
    0.12, 0.04,
    f"Top {top_n} countries shown separately, remaining grouped as 'Other'",
    fontsize=12, color="#666666"
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================
# 1) Filter Chinese vessels
# =========================================
df_china = df_taiwan[df_taiwan["flag"] == "CHN"].copy()

# =========================================
# 2) Unique boats per EEZ
# =========================================
boats_by_eez = (
    df_china.groupby("eez_iso")["mmsi_present"]
    .nunique()
)

total_boats = df_china["mmsi_present"].nunique()

# =========================================
# 3) Split: China EEZ vs Other EEZs
# =========================================
china_eez = boats_by_eez.get("CHN", 0)
foreign_eez = total_boats - china_eez

pct_china = 100 * china_eez / total_boats
pct_foreign = 100 * foreign_eez / total_boats

print(f"Chinese boats in Chinese EEZ: {pct_china:.1f}%")
print(f"Chinese boats in foreign EEZ: {pct_foreign:.1f}%")

In [ ]:
main_eez_per_boat = (
    df_china.groupby(["mmsi_present", "eez_iso"])
    .size()
    .reset_index(name="count")
    .sort_values(["mmsi_present", "count"], ascending=[True, False])
    .drop_duplicates("mmsi_present")
)

boats_by_eez = main_eez_per_boat["eez_iso"].value_counts()
pct_by_eez = boats_by_eez / boats_by_eez.sum() * 100

display(pct_by_eez.head(10))

In [ ]:
countries = ["CHN", "TWN", "KOR", "JPN"]

results = []

for country in countries:
    df_c = df_taiwan[df_taiwan["flag"] == country]

    total_boats = df_c["mmsi_present"].nunique()

    # Boats per EEZ
    boats_by_eez = (
        df_c.groupby("eez_iso")["mmsi_present"]
        .nunique()
    )

    domestic = boats_by_eez.get(country, 0)
    foreign = total_boats - domestic

    pct_foreign = 100 * foreign / total_boats if total_boats > 0 else 0

    results.append({
        "country": country,
        "pct_foreign": pct_foreign
    })

df_plot = pd.DataFrame(results).sort_values("pct_foreign", ascending=False)

# =========================================
# Plot
# =========================================
plt.figure(figsize=(8, 5))

bars = plt.bar(df_plot["country"], df_plot["pct_foreign"])

# Add labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 1,
             f"{height:.1f}%", ha="center", fontsize=11)

plt.title("Share of Boats Operating in Foreign EEZs", fontsize=16, weight="bold")
plt.ylabel("% of boats")
plt.xlabel("Country")

plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()